# Symbolic task planning: STRIPS and PDDL

The task of the Note: carry the cup from the table to the door. This notebook builds the facts and the 12 STRIPS actions, applies the plan, counts the frame statements, runs breadth-first and A* forward search, and finds the grid routes of the two move actions. The model lives in `images/taskplan.py`, which the figures use too.

In [1]:
import sys
sys.path.insert(0, 'images')
from taskplan import (FACTS, INIT, GOAL, ACTIONS, apply, applicable, successors,
                      reachable, bfs, astar, h_actions_left, short, grid_route, bfs_route4, CELL, BLOCKED)
print(len(FACTS), 'facts:', FACTS)
print('possible sets of facts:', 2 ** len(FACTS))
print('start state:', sorted(INIT))
print('goal:', sorted(GOAL))

8 facts: ['robot-at start', 'robot-at table', 'robot-at door', 'cup-at start', 'cup-at table', 'cup-at door', 'holding-cup', 'hand-empty']
possible sets of facts: 256
start state: ['cup-at table', 'hand-empty', 'robot-at start']
goal: ['cup-at door']


## The 12 STRIPS actions
Each action: precondition, add list, delete list.

In [2]:
for name, pre, add, de in ACTIONS:
    print(f'{name:18s} pre {sorted(pre)}  add {sorted(add)}  del {sorted(de)}')
print(len(ACTIONS), 'actions')

move start table   pre ['robot-at start']  add ['robot-at table']  del ['robot-at start']
move start door    pre ['robot-at start']  add ['robot-at door']  del ['robot-at start']
move table start   pre ['robot-at table']  add ['robot-at start']  del ['robot-at table']
move table door    pre ['robot-at table']  add ['robot-at door']  del ['robot-at table']
move door start    pre ['robot-at door']  add ['robot-at start']  del ['robot-at door']
move door table    pre ['robot-at door']  add ['robot-at table']  del ['robot-at door']
pick-up start      pre ['cup-at start', 'hand-empty', 'robot-at start']  add ['holding-cup']  del ['cup-at start', 'hand-empty']
pick-up table      pre ['cup-at table', 'hand-empty', 'robot-at table']  add ['holding-cup']  del ['cup-at table', 'hand-empty']
pick-up door       pre ['cup-at door', 'hand-empty', 'robot-at door']  add ['holding-cup']  del ['cup-at door', 'hand-empty']
put-down start     pre ['holding-cup', 'robot-at start']  add ['cup-at start', 'ha

## Applying an action: pick-up table in the state S1

In [3]:
act = {a[0]: a for a in ACTIONS}
S1 = apply(INIT, act['move start table'])
print('S1:', sorted(S1))
print('pick-up table applicable?', applicable(S1, act['pick-up table']))
S3 = apply(S1, act['pick-up table'])
print('after pick-up table:', sorted(S3))

S1: ['cup-at table', 'hand-empty', 'robot-at table']
pick-up table applicable? True
after pick-up table: ['holding-cup', 'robot-at table']


## The plan, applied state by state

In [4]:
s = INIT
for n in ['move start table', 'pick-up table', 'move table door', 'put-down door']:
    assert applicable(s, act[n])
    s = apply(s, act[n])
    print(f'{n:18s} ->', sorted(s))
print('goal holds:', GOAL <= s)

move start table   -> ['cup-at table', 'hand-empty', 'robot-at table']
pick-up table      -> ['holding-cup', 'robot-at table']
move table door    -> ['holding-cup', 'robot-at door']
put-down door      -> ['cup-at door', 'hand-empty', 'robot-at door']
goal holds: True


## The frame problem in numbers
Without the STRIPS rule, each action must say what every fact is afterwards.

In [5]:
changes = sum(len(a[2]) + len(a[3]) for a in ACTIONS)
all_lines = len(ACTIONS) * len(FACTS)
print('lines for every fact of every action:', all_lines)
print('STRIPS list entries:', changes)
print('lines that only say "stays the same":', all_lines - changes)
print('pick-up table changes', len(act['pick-up table'][2]) + len(act['pick-up table'][3]), 'of', len(FACTS), 'facts')

lines for every fact of every action: 96
STRIPS list entries: 30
lines that only say "stays the same": 66
pick-up table changes 3 of 8 facts


## Reachable states and forward search

In [6]:
print('10 facts:', 2 ** 10, 'sets;  move actions for 3 and 10 places:', 3 * 2, 10 * 9)

10 facts: 1024 sets;  move actions for 3 and 10 places: 6 90


In [7]:
R = reachable()
print(len(R), 'states reachable from the start, out of', 2 ** len(FACTS))
plan, log, parent = bfs()
print('breadth-first plan:', plan)
print('states taken from the frontier:', len(log), ' states made:', len(parent))
print('branching at the start:', [n for n, _ in successors(INIT)])

12 states reachable from the start, out of 256
breadth-first plan: ['move start table', 'pick-up table', 'move table door', 'put-down door']
states taken from the frontier: 8  states made: 10
branching at the start: ['move start table', 'move start door']


In [8]:
print('h at the start state:', h_actions_left(INIT))
plan2, closed = astar()
print('A* plan:', plan2)
print('states taken by A*:', len(closed), [short(s) for s in closed])

h at the start state: 4
A* plan: ['move start table', 'pick-up table', 'move table door', 'put-down door']
states taken by A*: 5 [('start', 'table'), ('table', 'table'), ('table', 'held'), ('door', 'held'), ('door', 'door')]


## The move actions as routes on the 0.5 m grid
Breadth-first search with 4-connected moves (the Overview's 8 moves), then shortest routes with diagonal moves that may not cut the corner of a blocked cell (Dijkstra; A* gives the same length).

In [9]:
print('blocked cells:', BLOCKED.sum(), ' free cells:', (~BLOCKED).sum())
r4 = bfs_route4(CELL['start'], CELL['door'])
print('breadth-first, 4-connected, start -> door:', len(r4) - 1, 'moves,', 0.5 * (len(r4) - 1), 'm')
for a, b in [('start', 'table'), ('table', 'door'), ('start', 'door')]:
    d, r = grid_route(CELL[a], CELL[b])
    print(f'{a} -> {b}: {d:.3f} m')

blocked cells: 36  free cells: 76
breadth-first, 4-connected, start -> door: 8 moves, 4.0 m
start -> table: 0.707 m
table -> door: 2.707 m
start -> door: 3.414 m


## The PDDL files
The domain file and the problem file of the Note, in `data/`.

In [10]:
for f in ['data/cup-domain.pddl', 'data/cup-problem.pddl', 'data/cup-problem-back.pddl']:
    print(open(f).read())

(define (domain cup-room)
  (:requirements :strips)
  (:predicates (robot-at ?p) (cup-at ?p) (holding-cup) (hand-empty))

  (:action move
    :parameters (?from ?to)
    :precondition (robot-at ?from)
    :effect (and (robot-at ?to) (not (robot-at ?from))))

  (:action pick-up
    :parameters (?p)
    :precondition (and (robot-at ?p) (cup-at ?p) (hand-empty))
    :effect (and (holding-cup) (not (cup-at ?p)) (not (hand-empty))))

  (:action put-down
    :parameters (?p)
    :precondition (and (robot-at ?p) (holding-cup))
    :effect (and (cup-at ?p) (hand-empty) (not (holding-cup)))))

(define (problem cup-to-door)
  (:domain cup-room)
  (:objects start table door)
  (:init (robot-at start) (cup-at table) (hand-empty))
  (:goal (cup-at door)))

(define (problem cup-back-to-table)
  (:domain cup-room)
  (:objects start table door)
  (:init (robot-at start) (cup-at door) (hand-empty))
  (:goal (cup-at table)))



## A planner reading the two files
Read each PDDL file as nested brackets, fill every action's parameters with the objects, and run breadth-first forward search, as a PDDL planner does.

In [11]:
from collections import deque
from itertools import product
from pyparsing import nested_expr


def read(path):
    """A PDDL file as nested lists: (a (b c)) -> ['a', ['b', 'c']]."""
    return nested_expr().parse_string(open(path).read()).as_list()[0]


def facts(e):
    """A condition or effect -> (facts made true, facts inside not)."""
    parts = e[1:] if e[0] == 'and' else [e]
    return [p for p in parts if p[0] != 'not'], [p[1] for p in parts if p[0] == 'not']


def solve(domain_file, problem_file):
    dom, prob = read(domain_file), read(problem_file)
    sec = {e[0]: e[1:] for e in prob if isinstance(e, list)}
    objects = sec[':objects']
    acts = []
    for a in (e for e in dom if isinstance(e, list) and e[0] == ':action'):
        d = dict(zip(a[2::2], a[3::2]))
        for vals in product(objects, repeat=len(d[':parameters'])):
            sub = dict(zip(d[':parameters'], vals))
            name = lambda f: ' '.join(sub.get(t, t) for t in f)
            pre, _ = facts(d[':precondition'])
            add, de = facts(d[':effect'])
            acts.append((' '.join([a[1], *vals]), {name(f) for f in pre}, {name(f) for f in add}, {name(f) for f in de}))
    init = frozenset(' '.join(f) for f in sec[':init'])
    goal = {' '.join(f) for f in facts(sec[':goal'][0])[0]}
    parent, q = {init: None}, deque([init])
    while q:
        s = q.popleft()
        if goal <= s:
            break
        for n, pre, add, de in acts:
            if pre <= s:
                t = (s - de) | add
                if t not in parent:
                    parent[t] = (s, n)
                    q.append(t)
    plan = []
    while parent[s]:
        s, n = parent[s]
        plan.append(n)
    return plan[::-1]


p1 = solve('data/cup-domain.pddl', 'data/cup-problem.pddl')
p2 = solve('data/cup-domain.pddl', 'data/cup-problem-back.pddl')
assert p1 == plan
print('cup-problem.pddl:     ', p1)
print('cup-problem-back.pddl:', p2)


cup-problem.pddl:      ['move start table', 'pick-up table', 'move table door', 'put-down door']
cup-problem-back.pddl: ['move start door', 'pick-up door', 'move door table', 'put-down table']
